``DesignConstraints`` is the shared container of design limits that :class:`AAMut`, :class:`SeqMut` and :class:`SeqOpt` all accept as ``constraints``. :meth:`DesignConstraints.check` is its primary contract: it answers ``(ok, reasons)`` for one candidate sequence and names every violated limit, so a rejected candidate explains itself. Positions are 1-based over the parent (wild-type) sequence, the same convention as ``region`` and the ``pos`` column of the mutation tables.

In [1]:
import pandas as pd
import aaanalysis as aa
aa.options["verbose"] = False

# One protein as the parent (wild-type) sequence
df_seq = aa.load_dataset(name="DOM_GSEC", n=5)
seq = df_seq["sequence"].iloc[0]
tmd_start, tmd_stop = int(df_seq["tmd_start"].iloc[0]), int(df_seq["tmd_stop"].iloc[0])
pos_a, pos_b, pos_c = tmd_start + 3, tmd_start + 6, tmd_start + 9

def mutate(sequence, pos, to_aa):
    """Substitute one 1-based position."""
    return sequence[:pos - 1] + to_aa + sequence[pos:]

# Position- and residue-level limits
dc = aa.DesignConstraints(immutable_positions=[tmd_start, tmd_stop],
                          permitted_substitutions=["A", "L", "V", "I"],
                          forbidden_substitutions={pos_b: ["V"]},
                          parent=seq)
candidates = {"single": mutate(seq, pos_a, "A"),
              "double": mutate(mutate(seq, pos_a, "A"), pos_b, "L"),
              "anchor": mutate(seq, tmd_start, "A"),
              "banned": mutate(seq, pos_b, "V")}
rows = []
for name, candidate in candidates.items():
    ok, reasons = dc.check(candidate=candidate)
    rows.append(dict(variant=name, is_feasible=ok, reasons="; ".join(reasons) or "-"))
df_check = pd.DataFrame(rows)
aa.display_df(df_check, n_rows=10, show_shape=True)

DataFrame shape: (4, 3)


,variant,is_feasible,reasons
1,single,True,-
2,double,True,-
3,anchor,False,immutable_posit...] are immutable
4,banned,False,forbidden_subst...] are forbidden


### Further parameters: `mutable_positions`, `n_mut_max`, `min_identity`, `max_identity`, `forbidden_motifs`, `required_motifs`

The limits above are per position and per residue. The remaining constructor parameters judge the candidate as a whole: `mutable_positions` is the span a substitution may fall in (an explicit position list here, since a part name such as `'tmd'` is applied by the calling class), `n_mut_max` is the mutation budget, `min_identity` / `max_identity` bound how close the candidate stays to its parent, and `forbidden_motifs` / `required_motifs` are literal substrings it must avoid or keep.

In [2]:
# Whole-candidate limits (the TMD interior is mutable, the flanks are not)
dc = aa.DesignConstraints(mutable_positions=list(range(tmd_start + 1, tmd_stop)),
                          n_mut_max=2,
                          min_identity=0.95,
                          max_identity=0.999,
                          forbidden_motifs=["WW"],
                          required_motifs=[seq[tmd_start - 1:tmd_start + 2]],
                          parent=seq)
candidates["triple"] = mutate(mutate(mutate(seq, pos_a, "A"), pos_b, "L"), pos_c, "V")
candidates["parent"] = seq
rows = []
for name, candidate in candidates.items():
    ok, reasons = dc.check(candidate=candidate)
    rows.append(dict(variant=name, n_reasons=len(reasons), is_feasible=ok,
                     reasons="; ".join(reasons) or "-"))
df_check = pd.DataFrame(rows)
aa.display_df(df_check, n_rows=10, show_shape=True)

DataFrame shape: (6, 4)


,variant,n_reasons,is_feasible,reasons
1,single,0,True,-
2,double,0,True,-
3,anchor,2,False,mutable_positio...m the candidate
4,banned,0,True,-
5,triple,1,False,n_mut_max: 3 mu...he maximum of 2
6,parent,1,False,max_identity: i...ximum of 0.9990


`parent` is available twice: set once on the object (above) or passed per call, which lets one constraint set be reused across proteins. A per-call `parent` overrides the stored one.

In [3]:
# No parent on the object: it is supplied per check() call
dc = aa.DesignConstraints(n_mut_max=1)
seq_other = df_seq["sequence"].iloc[1]
rows = []
for name, candidate in candidates.items():
    ok, reasons = dc.check(candidate=candidate, parent=seq)
    rows.append(dict(variant=name, parent="first protein", is_feasible=ok,
                     reasons="; ".join(reasons) or "-"))
ok, reasons = dc.check(candidate=mutate(seq_other, 5, "A"), parent=seq_other)
rows.append(dict(variant="single", parent="second protein", is_feasible=ok,
                 reasons="; ".join(reasons) or "-"))
df_check = pd.DataFrame(rows)
aa.display_df(df_check, n_rows=10, show_shape=True)

DataFrame shape: (7, 4)


,variant,parent,is_feasible,reasons
1,single,first protein,True,-
2,double,first protein,False,n_mut_max: 2 mu...he maximum of 1
3,anchor,first protein,True,-
4,banned,first protein,True,-
5,triple,first protein,False,n_mut_max: 3 mu...he maximum of 1
6,parent,first protein,True,-
7,single,second protein,True,-
